In [1]:
%pip install pyodbc sqlalchemy pandas

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.4 MB 4.4 MB/s eta 0:00:01
   ------------------------------ --------- 1.8/2.4 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.2 MB/s  0:00:00

   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   -----------------------

In [2]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# بيانات الاتصال بالسيرفر الخاص بك
server = r'DESKTOP-M66E32B\SQLEXPRESS'
database = 'CairoMetroDB'

params = urllib.parse.quote_plus(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# سحب الجداول الستة من SQL Server
lines_df = pd.read_sql("SELECT * FROM Lines", engine)
stations_df = pd.read_sql("SELECT * FROM Stations", engine)
station_lines_df = pd.read_sql("SELECT * FROM StationLines", engine)
trips_df = pd.read_sql("SELECT * FROM TrainTrips", engine)
flow_df = pd.read_sql("SELECT * FROM PassengerFlow", engine)
incidents_df = pd.read_sql("SELECT * FROM Incidents", engine)



In [5]:
lines_df.info 
lines_df.info()

lines_df.isnull().sum()

lines_df.duplicated().sum()

lines_df['LineID'].duplicated().sum()

for col in ['LineNameArabic', 'LineColor', 'StartStationArabic', 'EndStationArabic']:
    print(f"Unique values in {col}:", lines_df[col].unique())

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   LineID              3 non-null      int64
 1   LineNameArabic      3 non-null      str  
 2   LineColor           3 non-null      str  
 3   StartStationArabic  3 non-null      str  
 4   EndStationArabic    3 non-null      str  
dtypes: int64(1), str(4)
memory usage: 252.0 bytes
Unique values in LineNameArabic: <StringArray>
['الخط الأول', 'الخط الثاني', 'الخط الثالث']
Length: 3, dtype: str
Unique values in LineColor: <StringArray>
['Blue', 'Red', 'Green']
Length: 3, dtype: str
Unique values in StartStationArabic: <StringArray>
['حلوان', 'شبرا الخيمة', 'عدلي منصور']
Length: 3, dtype: str
Unique values in EndStationArabic: <StringArray>
['المرج الجديدة', 'المنيب', 'روض الفرج / جامعة القاهرة']
Length: 3, dtype: str


In [6]:
station_lines_df.info()

print(station_lines_df.isnull().sum())

print(station_lines_df.duplicated().sum())

print(station_lines_df.duplicated(subset=['LineID', 'StationID']).sum())

print(station_lines_df['BranchName'].unique())

<class 'pandas.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   LineID            89 non-null     int64
 1   StationID         89 non-null     int64
 2   StationOrder      89 non-null     int64
 3   BranchName        89 non-null     str  
 4   BranchNameArabic  89 non-null     str  
dtypes: int64(3), str(2)
memory usage: 3.6 KB
LineID              0
StationID           0
StationOrder        0
BranchName          0
BranchNameArabic    0
dtype: int64
0
0
<StringArray>
['Main', 'Cairo University', 'Rod El-Farag']
Length: 3, dtype: str


In [7]:
trips_df.info()

print(trips_df.isnull().sum())

print(trips_df.duplicated().sum())

print(trips_df.duplicated(subset=['TripID']).sum())

print(trips_df['TimePeriod'].unique())

print(trips_df[['ScheduledDurationMinutes', 'ActualDurationMinutes', 'DelayMinutes']].describe())

<class 'pandas.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   TripID                    1620 non-null   int64 
 1   LineID                    1620 non-null   int64 
 2   TripDate                  1620 non-null   object
 3   TimePeriod                1620 non-null   str   
 4   DepartureTime             1620 non-null   object
 5   ScheduledDurationMinutes  1620 non-null   int64 
 6   ActualDurationMinutes     1620 non-null   int64 
 7   DelayMinutes              1620 non-null   int64 
dtypes: int64(5), object(2), str(1)
memory usage: 101.4+ KB
TripID                      0
LineID                      0
TripDate                    0
TimePeriod                  0
DepartureTime               0
ScheduledDurationMinutes    0
ActualDurationMinutes       0
DelayMinutes                0
dtype: int64
0
0
<StringArray>
['Morning', 'Afternoon', 'Evening'

In [8]:
trips_df[['TripID', 'LineID', 'ScheduledDurationMinutes', 'ActualDurationMinutes', 'DelayMinutes']].head(5)

,TripID,LineID,ScheduledDurationMinutes,ActualDurationMinutes,DelayMinutes
0,1,1,78,82,4
1,2,1,78,77,1
2,3,1,78,78,3
3,4,1,78,83,8
4,5,1,78,85,3


In [9]:
flow_df.info()

print(flow_df.isnull().sum())

print(flow_df.duplicated().sum())

print(flow_df.duplicated(subset=['RecordID']).sum())

print(flow_df['TimePeriod'].unique())

display(flow_df[['PassengerCount']].describe())

<class 'pandas.DataFrame'>
RangeIndex: 8010 entries, 0 to 8009
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   RecordID        8010 non-null   int64 
 1   StationID       8010 non-null   int64 
 2   LineID          8010 non-null   int64 
 3   Date            8010 non-null   object
 4   TimePeriod      8010 non-null   str   
 5   PassengerCount  8010 non-null   int64 
dtypes: int64(4), object(1), str(1)
memory usage: 375.6+ KB
RecordID          0
StationID         0
LineID            0
Date              0
TimePeriod        0
PassengerCount    0
dtype: int64
0
0
<StringArray>
['Morning', 'Afternoon', 'Evening']
Length: 3, dtype: str


,PassengerCount
count,8010.000000
mean,1869.033708
std,1009.705498
min,292.000000
25%,1133.000000
50%,1635.000000
75%,2375.000000
max,10791.000000


In [10]:
incidents_df.info()

print(incidents_df.isnull().sum())

print(incidents_df.duplicated().sum())

print(incidents_df['IncidentType'].unique())

print(incidents_df['Severity'].unique())

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   IncidentID    500 non-null    int64 
 1   StationID     500 non-null    int64 
 2   IncidentDate  500 non-null    object
 3   IncidentTime  500 non-null    object
 4   IncidentType  500 non-null    str   
 5   Severity      500 non-null    str   
dtypes: int64(2), object(2), str(2)
memory usage: 23.6+ KB
IncidentID      0
StationID       0
IncidentDate    0
IncidentTime    0
IncidentType    0
Severity        0
dtype: int64
0
<StringArray>
['مساعدة ركاب', 'مشكلة فنية', 'ازدحام', 'مشكلة مرافق', 'تأخير خدمة']
Length: 5, dtype: str
<StringArray>
['Medium', 'High', 'Low']
Length: 3, dtype: str
